# TopoSurface-DTI — Kaggle Notebook (Real PDBBind Data)

Trains the full TopoSurface-DTI pipeline on real PDBBind protein-ligand binding affinity data.

**Before running, attach two datasets to this notebook:**

1. **Code dataset:** `drug-target-gdl`  
   (Add Data → Your Datasets → drug-target-gdl)

2. **PDBBind dataset:** `pdbbind-protein-ligand-binding-affinity-dataset` by madukacharles  
   (Add Data → Search `pdbbind-protein-ligand-binding-affinity-dataset` → madukacharles)

In [ ]:
# ── 1. Install missing dependencies ──────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ripser', 'pyyaml'])
print('Dependencies installed.')

In [ ]:
# ── 1b. Install real-data dependencies ───────────────────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'biopython', 'rdkit-pypi'])
print('Real-data dependencies installed.')

In [ ]:
# ── 2. Add project to Python path ────────────────────────────────────────
import sys, os

PROJECT_DIR = '/kaggle/input/drug-target-gdl/Drug_target_GDL'
if not os.path.isdir(PROJECT_DIR):
    PROJECT_DIR = '/kaggle/input/drug-target-gdl'

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files found:', os.listdir('.'))

In [ ]:
# ── 2b. Generate train/val/test split JSON files ─────────────────────────
import subprocess, sys, os

DATA_DIR  = '/kaggle/input/datasets/madukacharles/pdbbind-protein-ligand-binding-affinity-dataset'
SPLIT_DIR = '/kaggle/working'

result = subprocess.run(
    [sys.executable, 'scripts/make_splits.py',
     '--data_dir', DATA_DIR,
     '--out_dir',  SPLIT_DIR],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Split generation failed — check the output above.')

In [ ]:
# ── 3. Verify environment ─────────────────────────────────────────────────
import torch, numpy as np
from ripser import ripser

print(f'PyTorch : {torch.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU run)"}')

pts  = np.random.default_rng(0).uniform(0, 5, (10, 3))
dgms = ripser(pts, maxdim=1)['dgms']
print(f'Ripser  : OK  (H0={len(dgms[0])} bars, H1={len(dgms[1])} bars)')

In [ ]:
# ── 4. Forward-pass smoke test ────────────────────────────────────────────
import numpy as np
from data.molecule_graph    import synthetic_drug_graph
from data.pocket_mesh       import synthetic_pocket_graph
from data.tda_features      import compute_tda_features, tda_to_tensor
from models.toposurface_dti import TopoSurfaceDTI

drug   = synthetic_drug_graph(n_atoms=20,     seed=0)
pocket = synthetic_pocket_graph(n_residues=30, seed=0)

drug_tda   = tda_to_tensor(compute_tda_features(drug['pos'].numpy(),   max_edge_len=8.0))
pocket_tda = tda_to_tensor(compute_tda_features(pocket['pos'].numpy(), max_edge_len=16.0))

model  = TopoSurfaceDTI()
params = model.count_parameters()
print('Parameter counts:')
for k, v in params.items():
    print(f'  {k:20s}: {v:,}')

pred = model(
    drug_x=drug['x'], drug_pos=drug['pos'], drug_edge=drug['edge_index'],
    pocket_x=pocket['x'], pocket_edge=pocket['edge_index'],
    pocket_angles=pocket['angles'], pocket_trans=pocket['transporters'],
    drug_tda=drug_tda, pocket_tda=pocket_tda,
)
print(f'\nForward pass: pKd = {pred.item():.4f}  ✓')

In [ ]:
# ── 5. Training on real PDBBind data ─────────────────────────────────────
import yaml, os, torch, torch.nn as nn, numpy as np
from torch.utils.data import DataLoader
from data.dataset       import DTIDataset
from train.trainer      import collate_single, train_epoch, validate
from models.toposurface_dti import TopoSurfaceDTI

DATA_DIR  = '/kaggle/input/datasets/madukacharles/pdbbind-protein-ligand-binding-affinity-dataset'
SPLIT_DIR = '/kaggle/working'

with open('configs/base.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['use_synthetic'] = False
cfg['n_epochs']      = 50
cfg['lr']            = 1e-3

# DTIDataset uses data_dir for both split JSONs and structure files.
# On Kaggle these live in different locations, so we subclass and redirect
# _load_real to the read-only input directory.
class PDBBindDataset(DTIDataset):
    def __init__(self, structure_dir, split_dir, split, **kwargs):
        self.structure_dir = structure_dir
        super().__init__(data_dir=split_dir, split=split, use_synthetic=False, **kwargs)

    def _load_real(self, pdb_id):
        from data.molecule_graph import mol_to_graph, synthetic_drug_graph
        from data.pocket_mesh    import (extract_pocket_atoms, build_knn_graph,
                                         precompute_pocket_geometry, synthetic_pocket_graph)
        try:
            from rdkit import Chem
            sdf = os.path.join(self.structure_dir, pdb_id, f'{pdb_id}_ligand.sdf')
            mol = Chem.SDMolSupplier(sdf, removeHs=True)[0]
            drug = mol_to_graph(mol)
        except Exception as e:
            print(f'[warn] Drug {pdb_id}: {e}')
            drug = synthetic_drug_graph()

        try:
            pdb = os.path.join(self.structure_dir, pdb_id, f'{pdb_id}_protein.pdb')
            sdf = os.path.join(self.structure_dir, pdb_id, f'{pdb_id}_ligand.sdf')
            pos_np, feat_np = extract_pocket_atoms(
                pdb, cutoff_angstrom=self.pocket_cutoff,
                sdf_path=sdf if os.path.exists(sdf) else None,
            )
            pos_t  = torch.from_numpy(pos_np)
            feat_t = torch.from_numpy(feat_np)
            ei     = torch.from_numpy(build_knn_graph(pos_np, k=self.knn_k))
            geo    = precompute_pocket_geometry(pos_t, ei)
            pocket = {'x': feat_t, 'pos': pos_t, 'edge_index': ei, **geo}
        except Exception as e:
            print(f'[warn] Pocket {pdb_id}: {e}')
            pocket = synthetic_pocket_graph()

        return drug, pocket

train_ds = PDBBindDataset(DATA_DIR, SPLIT_DIR, 'train', tda_resolution=cfg['tda_resolution'])
val_ds   = PDBBindDataset(DATA_DIR, SPLIT_DIR, 'val',   tda_resolution=cfg['tda_resolution'])
print(f'Train samples: {len(train_ds)} | Val samples: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=collate_single)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, collate_fn=collate_single)

# Target normalisation: compute mean/std from training affinities
train_affinities = torch.tensor([train_ds[i]['affinity'].item() for i in range(len(train_ds))])
target_mean = float(train_affinities.mean())
target_std  = float(train_affinities.std().clamp(min=1e-3))
print(f'Target  mean={target_mean:.3f}  std={target_std:.3f}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model     = TopoSurfaceDTI.from_config(cfg).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=30, min_lr=1e-5)
loss_fn   = nn.MSELoss()

os.makedirs('checkpoints', exist_ok=True)
best_val_rmse = float('inf')
accum_steps   = 16

for epoch in range(cfg['n_epochs']):
    tr = train_epoch(model, train_loader, optimizer, device, loss_fn, target_mean, target_std, accum_steps)
    va = validate(model, val_loader, device, loss_fn, target_mean, target_std)
    scheduler.step(va['loss'])
    print(f"Epoch {epoch:03d} | Train loss={tr['loss']:.4f} RMSE={tr['rmse']:.3f} R={tr['r']:.3f} | "
          f"Val loss={va['loss']:.4f} RMSE={va['rmse']:.3f} R={va['r']:.3f}")
    if va['rmse'] < best_val_rmse:
        best_val_rmse = va['rmse']
        torch.save({
            'epoch':       epoch,
            'model':       model.state_dict(),
            'cfg':         cfg,
            'target_mean': target_mean,
            'target_std':  target_std,
        }, 'checkpoints/best_model.pt')

print(f'\nTraining complete. Best Val RMSE: {best_val_rmse:.4f}')

In [ ]:
# ── 6. Visualise: Predicted vs Actual ────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from torch.utils.data import DataLoader

from train.trainer import collate_single, forward_step

# Load target normalisation stats saved during training
ckpt        = torch.load('checkpoints/best_model.pt', map_location='cpu')
target_mean = ckpt.get('target_mean', 0.0)
target_std  = ckpt.get('target_std',  1.0)

device = next(model.parameters()).device
model.eval()
loss_fn = torch.nn.MSELoss()

loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_single)

preds, actuals = [], []
with torch.no_grad():
    for sample in loader:
        p, _ = forward_step(model, sample, device, loss_fn, target_mean, target_std)
        preds.append(float(p.cpu()))
        actuals.append(float(sample['affinity']))

preds   = np.array(preds)
actuals = np.array(actuals)
errors  = preds - actuals
abs_err = np.abs(errors)

rmse_val    = float(np.sqrt(np.mean(errors**2)))
mae         = float(np.mean(abs_err))
pearson_r,  _ = stats.pearsonr(preds, actuals)
spearman_r, _ = stats.spearmanr(preds, actuals)
bias        = float(np.mean(errors))
std_err     = float(np.std(errors))

print(f'n={len(preds)}  RMSE={rmse_val:.3f}  MAE={mae:.3f}  R={pearson_r:.3f}  ρ={spearman_r:.3f}')

In [ ]:
# ── 4-panel figure ───────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(14, 11))
fig.suptitle(
    f'TopoSurface-DTI  ·  Predicted vs Actual Binding Affinity  ·  n={len(preds)} compounds',
    fontsize=13, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32,
                       left=0.07, right=0.96, top=0.93, bottom=0.07)

# ── Panel 1: Scatter ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
sc  = ax1.scatter(actuals, preds, c=abs_err, cmap='RdYlGn_r', s=40,
                  alpha=0.75, edgecolors='none',
                  vmin=0, vmax=np.percentile(abs_err, 95))
lo  = min(actuals.min(), preds.min()) - 0.3
hi  = max(actuals.max(), preds.max()) + 0.3
ax1.plot([lo, hi], [lo, hi], 'k--', lw=1.2, label='Perfect prediction')
ax1.fill_between([lo, hi], [lo-rmse_val, hi-rmse_val], [lo+rmse_val, hi+rmse_val],
                 color='steelblue', alpha=0.08, label=f'±RMSE band')
slope, intercept, *_ = stats.linregress(actuals, preds)
xs = np.linspace(lo, hi, 100)
ax1.plot(xs, slope*xs + intercept, 'steelblue', lw=1.5, alpha=0.8, label='Linear fit')
plt.colorbar(sc, ax=ax1, label='|Error| (pKd)', shrink=0.85)
ax1.set(xlim=(lo,hi), ylim=(lo,hi),
        xlabel='Actual pKd', ylabel='Predicted pKd',
        title='Predicted vs Actual')
ax1.legend(fontsize=8, loc='upper left')
ax1.text(0.97, 0.05,
         f'RMSE = {rmse_val:.3f}\nMAE  = {mae:.3f}\nR    = {pearson_r:.3f}\nρ    = {spearman_r:.3f}',
         transform=ax1.transAxes, fontsize=9, va='bottom', ha='right',
         bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85, ec='#cccccc'))
ax1.set_title('Predicted vs Actual', fontsize=12, fontweight='bold')

# ── Panel 2: Residuals histogram ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
n_bins = max(10, len(errors) // 8)
ax2.hist(errors, bins=n_bins, color='steelblue', alpha=0.65,
         edgecolor='white', linewidth=0.5, density=True, label='Residuals')
xs_r = np.linspace(errors.min()-0.5, errors.max()+0.5, 200)
ax2.plot(xs_r, stats.norm.pdf(xs_r, bias, std_err), 'crimson', lw=2,
         label=f'N({bias:.2f}, {std_err:.2f}²)')
ax2.axvline(0,        color='black',   lw=1.2, ls='--', alpha=0.7)
ax2.axvline(bias,     color='crimson', lw=1.2, ls=':',  alpha=0.9, label=f'Bias={bias:.3f}')
ax2.axvline( std_err, color='grey',    lw=0.8, ls=':',  alpha=0.6)
ax2.axvline(-std_err, color='grey',    lw=0.8, ls=':',  alpha=0.6)
within_1 = float(np.mean(abs_err < 1.0) * 100)
ax2.text(0.97, 0.95, f'{within_1:.1f}% within ±1 pKd',
         transform=ax2.transAxes, fontsize=9, va='top', ha='right',
         bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85, ec='#cccccc'))
ax2.set(xlabel='Predicted − Actual (pKd)', ylabel='Density',
        title='Residuals Distribution')
ax2.legend(fontsize=8)
ax2.set_title('Residuals Distribution', fontsize=12, fontweight='bold')

# ── Panel 3: Ranking view ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
order        = np.argsort(actuals)
xs_rank      = np.arange(len(order))
act_sorted   = actuals[order]
pred_sorted  = preds[order]
ax3.plot(xs_rank, act_sorted,  color='#2c7bb6', lw=2,   label='Actual pKd')
ax3.plot(xs_rank, pred_sorted, color='#d7191c', lw=1.5, alpha=0.8, label='Predicted pKd')
ax3.fill_between(xs_rank, act_sorted, pred_sorted,
                 where=(pred_sorted >= act_sorted),
                 color='#d7191c', alpha=0.12, label='Over-prediction')
ax3.fill_between(xs_rank, act_sorted, pred_sorted,
                 where=(pred_sorted <  act_sorted),
                 color='#2c7bb6', alpha=0.12, label='Under-prediction')
ax3.set(xlabel='Compound rank (sorted by actual pKd)', ylabel='pKd',
        xlim=(0, len(xs_rank)-1))
ax3.legend(fontsize=8, ncol=2)
ax3.set_title('Ranking View  (Virtual Screening)', fontsize=12, fontweight='bold')

# ── Panel 4: Cumulative error distribution ────────────────────────────────
ax4  = fig.add_subplot(gs[1, 1])
ae_s = np.sort(abs_err)
cdf  = np.arange(1, len(ae_s)+1) / len(ae_s)
ax4.plot(ae_s, cdf*100, color='#1a9641', lw=2.5)
ax4.fill_between(ae_s, cdf*100, alpha=0.15, color='#1a9641')
for t, lbl in [(0.5,'0.5'), (1.0,'1.0'), (1.5,'1.5'), (2.0,'2.0')]:
    pct = float(np.mean(ae_s <= t)*100)
    ax4.axvline(t, color='grey', lw=0.8, ls='--', alpha=0.6)
    ax4.text(t+0.02, 5, f'{pct:.0f}%\n≤{lbl}', fontsize=7.5, color='grey', va='bottom')
ax4.set(xlabel='Absolute error (pKd)', ylabel='Cumulative % of predictions',
        xlim=(0, None), ylim=(0, 102))
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax4.grid(axis='y', alpha=0.3)
ax4.set_title('Cumulative Error Distribution', fontsize=12, fontweight='bold')

plt.savefig('/kaggle/working/predictions_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to /kaggle/working/predictions_vs_actual.png')

# ── Summary table ────────────────────────────────────────────────────────
print('\n── Evaluation Metrics ──────────────────────')
print(f'  RMSE         : {rmse_val:.4f} pKd')
print(f'  MAE          : {mae:.4f} pKd')
print(f'  Pearson R    : {pearson_r:.4f}')
print(f'  Spearman ρ   : {spearman_r:.4f}')
print(f'  Bias         : {bias:.4f} pKd')
print(f'  Error std    : {std_err:.4f} pKd')
for t in [0.5, 1.0, 1.5, 2.0]:
    print(f'  Within ±{t:.1f}   : {np.mean(abs_err<=t)*100:.1f}%')
print('────────────────────────────────────────────')